In [1]:
# goal: inspect embedding array shape in seo_5_toy.npz

from pathlib import Path
import json
import time
import numpy as np

LOG_PATH = Path("../.cursor/debug-bd1b15.log")

def _dbg(hypothesis_id, location, message, data):
    # #region agent log
    payload = {
        "sessionId": "bd1b15",
        "runId": "pre-fix",
        "hypothesisId": hypothesis_id,
        "location": location,
        "message": message,
        "data": data,
        "timestamp": int(time.time() * 1000),
    }
    with LOG_PATH.open("a") as f:
        f.write(json.dumps(payload) + "\n")
    # #endregion

# Notebook kernel cwd should be toy/
npz_path = Path("embeddings/seo_5_toy.npz")
_dbg("A", "toy.ipynb:cell0", "resolved path", {
    "cwd": str(Path.cwd()),
    "npz_path": str(npz_path),
    "resolved": str(npz_path.resolve()),
    "exists": npz_path.exists(),
})

data = np.load(npz_path)
emb = data["A_embeddings"]
_dbg("D", "toy.ipynb:cell0", "loaded embeddings", {"shape": list(emb.shape), "dtype": str(emb.dtype)})

print(f"Loading: {npz_path.resolve()}")
print(f"A_embeddings shape: {emb.shape}")


Loading: /Users/andrewlee/Documents/Keasling/Nathan_Kyro/PETase/PETase_structure_activity/toy/embeddings/seo_5_toy.npz
A_embeddings shape: (270, 1280)


In [2]:
""" 
- goal manipulate my sequence table in preparation for joining with npz files to create a toy feature table
- I will reorder the column in numerical order, then I will rename them, I will save this to replace the original Seo_tested_seq.csv
- then I will extract only the first 6 rows of this table to create a toy feature table

"""

#import the necessary stuff to do these tasks

import pandas as pd
from pathlib import Path

#load the table

df = pd.read_csv("../Data/Seo_data/seo_tested_seq.csv")

#reorder the table

df_sorted = df.sort_values("Library ID")

#output the edited table

df_sorted.to_csv("../Data/Seo_data/seo_tested_seq_sorted.csv", index=False)


FileNotFoundError: [Errno 2] No such file or directory: '../Data/Seo_data/seo_tested_seq.csv'

In [ ]:
"""
- now let's take the just the first 5 rows of our sorted seo_tested_seq_sorted.csv
bc those match the toy embedding npz things I have
"""

#define a new data frame that will read the larger overall csv

df_sorted_seq = pd.read_csv("../Data/Seo_data/seo_tested_seq_sorted.csv")

#now make a new data frame that only takes the top 5 rows and then turn that into a csv

df_first5 = df_sorted_seq.head(5)
df_first5.to_csv("seo_tested_seq_sorted_toy.csv")

In [ ]:
"""I'm gonna write some code that will rename all libary IDs to have "seo_" in front of them"""

##import pandas as pd

#open the csv

##df = pd.read_csv("seo_tested_seq_sorted_toy.csv", index_col =0)

#now use an f string to add seo_

##df["Library ID"] = df["Library ID"].apply(lambda x: f"seo_{x}")

#now save it

##df.to_csv("seo_tested_seq_sorted_toy.csv", index = False)

#alternatively I could have used a string df["Library ID"] = "seo_" + df["Library ID"].astype(str)



In [ ]:
"""I had to have cursor write this cell because my previous cell re-added seo everytime I"""

import pandas as pd

df = pd.read_csv("seo_tested_seq_sorted_toy.csv")

# strip ALL repeated "seo_" prefixes, then add exactly one
df["Library ID"] = df["Library ID"].astype(str).str.replace(r"^(?:seo_)+", "", regex=True)
df["Library ID"] = "seo_" + df["Library ID"]

df.to_csv("seo_tested_seq_sorted_toy.csv", index=False)
df["Library ID"]

0     seo_5
1     seo_8
2    seo_11
3    seo_13
4    seo_19
Name: Library ID, dtype: str

In [ ]:
"""code that I copied from cursor that will create a feature table for the toy npz file set
this is like a test of this concept, creating a csv feature table isn't necessary and will be huge
and is really ok only to do for like 5 proteins, I'm doing this because it's low effort but yeah anyways
let's do it"""

import pandas as pd
import numpy as np
from pathlib import Path

# your label/metadata table (all columns kept)
labels = pd.read_csv("seo_tested_seq_sorted_toy.csv", index_col=0)
labels["Library ID"] = labels["Library ID"].astype(int)

# build one row of features per npz file
feature_rows = []
for npz_path in sorted(Path("embeddings").glob("seo_*_toy.npz")):
    lib_id = int(npz_path.stem.split("_")[1])          # seo_5_toy -> 5
    emb = np.load(npz_path)["A_embeddings"]             # (270, 1280)
    protein_vec = emb.mean(axis=0)                       # (1280,)

    row = {"Library ID": lib_id}
    for i, val in enumerate(protein_vec):
        row[f"esm2_{i}"] = val
    feature_rows.append(row)

features = pd.DataFrame(feature_rows)

# glue features onto labels by Library ID
feature_table = labels.merge(features, on="Library ID")

feature_table.to_csv("seo_feature_table_toy.csv", index=False)
feature_table.shape   # should be (5, 5 + 1280) roughly

(5, 1285)

In [1]:
import sys
print(sys.executable)
# might show predict_p450 if that's the kernel

/opt/miniconda3/envs/petase_structure_activity/bin/python
